Data Preprocessing and standardization

In [6]:
import torch,torchvision,sklearn,albumentations,os
import pandas as pd
import cv2
from torch.utils.data import Dataset
from PIL import Image
import warnings
warnings.filterwarnings("ignore")
from tensorflow.keras.preprocessing.image import ImageDataGenerator

d:\proj\diabetic-retinopathy-detection\venv\lib\site-packages\albumentations\__init__.py:13: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.18). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


In [7]:
base_image_dir = './data/train_images/'

df = pd.read_csv('./data/train.csv')


# create a new column 'filename' that matches actual file on disk.
df['filename'] = df['id_code'].astype(str) + '.png'

#reduce the number of classes from 5 to 3 for better class balance
def reduce_classes(label):
    if label == 0: return 0        # Normal
    elif label <= 2: return 1      # Mild/Moderate
    else: return 2                 # Severe/Proliferative
df['diagnosis_3cat'] = df['diagnosis'].apply(reduce_classes)


print(df.head())

        id_code  diagnosis          filename  diagnosis_3cat
0  000c1434d8d7          2  000c1434d8d7.png               1
1  001639a390f0          4  001639a390f0.png               2
2  0024cdab0c1e          1  0024cdab0c1e.png               1
3  002c21358ce6          0  002c21358ce6.png               0
4  005b95c28852          0  005b95c28852.png               0


In [8]:
import numpy as np

def crop_image_from_gray(img, tol=7):
    """
    Crops the black borders using a threshold (tol).
    if the image is too dark, it returns the original.
    """
    if img.ndim == 2:
        mask = img > tol
        return img[np.ix_(mask.any(1), mask.any(0))]
    elif img.ndim == 3:
        gray_img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        mask = gray_img > tol
        
        check_shape = img[:,:,0][np.ix_(mask.any(1), mask.any(0))].shape[0]
        if (check_shape == 0): # Image is too dark, return original
            return img
        else:
            img1 = img[:,:,0][np.ix_(mask.any(1), mask.any(0))]
            img2 = img[:,:,1][np.ix_(mask.any(1), mask.any(0))]
            img3 = img[:,:,2][np.ix_(mask.any(1), mask.any(0))]
            img = np.stack([img1, img2, img3], axis=-1)
    return img

def circle_crop(img, sigmaX=10):
    """
    Step 1: Crop black borders
    Step 2: Resize to standard size (e.g., 224 or 256)
    Step 3: Apply Ben Graham's preprocessing method
    """
    # removing black borders
    img = crop_image_from_gray(img)
    
    # resizing to standard size
    img = cv2.resize(img, (224, 224))
    
    # Ben Graham's Method (Color + Gaussian Blur)
    # formula balances the brightness to a middle gray (128)
    # and subtracts the "local average" color (blur) to highlight edges.
    img = cv2.addWeighted(img, 4, cv2.GaussianBlur(img, (0,0), sigmaX), -4, 128)
    
    return img

In [9]:

class RetinopathyDataset(df):
    def __init__(self, dataframe, root_dir, transform=None):
        self.df = dataframe
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = os.path.join(self.root_dir, self.df.iloc[idx]['filename'])
        
        # Open with OpenCV for the preprocessing
        image = cv2.imread(img_name)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) # Fix color fix
        
        # applying ben graham's preprocessing
        image = circle_crop(image) 
        
        # convert to PIL for PyTorch transforms compatibility   
        image = Image.fromarray(image)

        if self.transform:
            image = self.transform(image)
        
        label = int(self.df.iloc[idx]['diagnosis_3cat'])
        return image, label

Now we start with Transfer training

Here  we are using transfer learning and the model ResNet50